In [1]:
import numpy as np
import scipy.constants
import scipy.sparse
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import math
import geopandas as gpd
import pandas as pd
import os
import netCDF4 as nc

In [2]:
def calc_alongshore_transport_k(gravity=scipy.constants.g, n=1.0, rho_water=1050.0, gamma_b=0.78, ):

    '''Calculates sediment transport constant k.'''
    
    return (
        5.3e-6
        # * 0.46
        * rho_water
        * gravity**1.5
        * (1 / (2 * n)) ** 1.2
        * (np.sqrt(gravity * gamma_b) / (2 * np.pi)) ** 0.2
    )

In [3]:
def circular_mean_2(a, b):
    """Circular mean of two angle DataArrays already in radians."""
    sin_mean = (np.sin(a) + np.sin(b)) / 2
    cos_mean = (np.cos(a) + np.cos(b)) / 2
    return np.arctan2(sin_mean, cos_mean)

In [4]:
def get_angles_xr(ds):
    
    ''' Takes average latitude and longitude calculated in beahc_w_to_gps.ipynb and computes bearing using bearing formula
    \(\theta =\>\mathrm{atan2}\>(\sin \Delta \lambda \cdot \cos \phi _{2},\cos \phi _{1}\cdot \sin \phi _{2}-\sin \phi _{1}\cdot \cos \phi _{2}\cdot \cos \Delta \lambda )\).'''
    
    lat = np.radians(ds['mean_lat'])
    lon = np.radians(ds['mean_lon'])

    lat1 = lat.shift(site=1)
    lon1 = lon.shift(site=1)
    lat2 = lat.shift(site=-1)
    lon2 = lon.shift(site=-1)

    dlon = lon2 - lon1
    
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)

    bearing = np.degrees(np.arctan2(y, x))
    bearing = (bearing + 360) % 360
    bearing_clean = bearing.fillna(0)

    return ds.assign(bearing=bearing_clean)

In [5]:
def calc_qs_mu(dp, hs, tp, angles, k, d=8): # mean direction of propagation bearing from north (degrees), significant wave height (meters), period (seconds), k, shoreface depth (meters) 

    '''Calculates Q_s and diffusivity (mu) using formula from Ashton, Murray 2006 B, equation (7) and (8). Negative sign is added so that positive Q_s 
    corresponds to rightward sediment movement when facing offshore (in this case, roughly south-bound).'''
    
    tp_da = tp['tp'] if isinstance(tp, xr.Dataset) else tp # just in case
    hs_da = hs['hs'] if isinstance(hs, xr.Dataset) else hs
    dp_da = dp['dp'] if isinstance(dp, xr.Dataset) else dp

    phi = dp_da - 90
    thet = ((angles + 90) % 360) - 90 # assumes no shoreline bearing angles equal to 270
    angle_term = ((phi - thet + 180) % 360) - 180
    angle_term = angle_term.where(np.abs(angle_term) <= 90) # mask areas where waves appear to be moving away from shore
    angle_term_abs = np.abs(angle_term)
    rad_factor = np.pi / 180
    angle_term = angle_term * rad_factor
    angle_term_abs = angle_term_abs * rad_factor
    cos_term = np.cos(angle_term)
    sin_term = np.sin(angle_term)
    sin_term_abs = np.sin(angle_term_abs)
    cos_term_abs = np.cos(angle_term_abs)
    power_term = ((tp_da * cos_term) ** 0.2) * (hs_da ** 2.4) 
    power_term_abs = ((tp_da * cos_term_abs) ** 0.2) * (hs_da ** 2.4)
    coeff = k / d
    mu = -(coeff * power_term_abs * ((1.2 * (sin_term_abs**2)) - (cos_term_abs**2)))
    qs = -k * (tp_da ** .2) * (hs_da ** 2.4) * (cos_term ** 1.2) * sin_term # CERC formula, negative fixes the bearing assumption so that downdrift qs is positive
    # qs = -k * (hs_da ** 2) * cos_term * sin_term # breaking wave formula
    return qs, mu, angle_term

In [6]:
def get_u_a(ds, start_date, end_date):

    '''Calculates U, the fraction of high angle wave contribution to Q_s over the total wave contribution to Q_s over the given time period.
    High-angle threshold is defined by the absolute angle between wave crests and the shoreline being greater than pi/4.'''

    '''Calculates A, the fraction of high rightward wave contribution to Q_s over the total wave contribution to Q_s over the given time period.'''
    
    ds_subset = ds.sel(time=slice(start_date, end_date))
    
    high_mask = (ds_subset['angle_diff'] > (np.pi/4)) | (ds_subset['angle_diff'] < -(np.pi/4))
    low_mask  = (ds_subset['angle_diff'] <= (np.pi/4)) & (ds_subset['angle_diff'] >= -(np.pi/4))
    qs_high = np.abs(ds_subset['qs'].where(high_mask)).sum(dim='time')
    qs_low  = np.abs(ds_subset['qs'].where(low_mask)).sum(dim='time')
    total1 = qs_low + qs_high
    u = (qs_high / total1)

    left_mask = (ds_subset['angle_diff'] < 0)
    right_mask = (ds_subset['angle_diff'] >= 0)
    qs_left = np.abs(ds_subset['qs'].where(left_mask)).sum(dim='time')
    qs_right  = np.abs(ds_subset['qs'].where(right_mask)).sum(dim='time')
    total2 = qs_left + qs_right
    a = (qs_left / total2)
    
    return u, a

In [7]:
def get_endpoint_bearing(ds, lat_min, lat_max, lat_var='mean_lat', lon_var='mean_lon'):

    """Bearing of the straight chord connecting the two endpoints of a shoreline domain.

    Returns the chord bearing (deg from north) and a diagnostics dict, including sinuosity:
    the summed point-to-point path length divided by the chord length. Values near 1 mean the
    reach is nearly straight and a single bearing is representative; larger values mean real
    curvature is being collapsed into one number.
    """

    sub = ds.where((ds[lat_var] >= lat_min) & (ds[lat_var] <= lat_max), drop=True)

    lat_vals = np.asarray(sub[lat_var].values, dtype=float)
    lon_vals = np.asarray(sub[lon_var].values, dtype=float)

    good = np.isfinite(lat_vals) & np.isfinite(lon_vals)
    if good.sum() < 2:
        raise ValueError(f'need >= 2 valid shoreline points in [{lat_min}, {lat_max}]')
    lat_vals, lon_vals = lat_vals[good], lon_vals[good]

    order = np.argsort(lat_vals)          # pin direction: south -> north
    i0, i1 = order[0], order[-1]

    lat1, lon1 = np.radians(lat_vals[i0]), np.radians(lon_vals[i0])
    lat2, lon2 = np.radians(lat_vals[i1]), np.radians(lon_vals[i1])
    dlon = lon2 - lon1

    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.degrees(np.arctan2(y, x)) % 360

    # haversine chord length, and path length along sites in latitude order
    R = 6371000.0
    def _hav(p1, l1, p2, l2):
        a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin((l2 - l1) / 2) ** 2
        return 2 * R * np.arcsin(np.sqrt(a))

    chord = _hav(lat1, lon1, lat2, lon2)
    plat, plon = np.radians(lat_vals[order]), np.radians(lon_vals[order])
    path = _hav(plat[:-1], plon[:-1], plat[1:], plon[1:]).sum()

    # circular mean of per-site bearings, for comparison
    mean_bearing = np.nan
    if 'bearing' in sub:
        b = np.radians(np.asarray(sub['bearing'].values, dtype=float))
        b = b[np.isfinite(b)]
        if b.size:
            mean_bearing = np.degrees(np.arctan2(np.sin(b).sum(), np.cos(b).sum())) % 360

    info = {
        'n_sites': int(lat_vals.size),
        'lat_start': float(lat_vals[i0]), 'lon_start': float(lon_vals[i0]),
        'lat_end':   float(lat_vals[i1]), 'lon_end':   float(lon_vals[i1]),
        'chord_km': chord / 1000.0,
        'path_km': path / 1000.0,
        'sinuosity': path / chord if chord > 0 else np.nan,
        'seaward_normal': (bearing + 90) % 360,
        'circular_mean_bearing': mean_bearing,
        'bearing_minus_mean': ((bearing - mean_bearing + 180) % 360) - 180,
    }
    return float(bearing), info

In [8]:
def calc_stats_for_bearing_single(dp, hs, tp, site_ids, bearing_val, time_windows, k, d=8):

    """Runs calc_qs_mu at a single shoreline bearing and summarises each site/time window.

    Reports mu_avg alongside the fraction of records that are antidiffusive and the ratio of
    negative to positive mu contributions, because mu_avg is a small residual of two large
    opposing terms and its sign alone is easy to over-read. n_valid counts unmasked records so
    an empty window is distinguishable from a genuine zero.
    """

    rows = []
    for site_id in site_ids:
        qs_da, mu_da, angle_da = calc_qs_mu(
            dp=dp.sel(site=site_id), hs=hs.sel(site=site_id), tp=tp.sel(site=site_id),
            angles=bearing_val, k=k, d=d,
        )
        ds_site = xr.Dataset({'qs': qs_da, 'angle_diff': angle_da, 'mu': mu_da})

        for start, end in time_windows:
            sub = ds_site.sel(time=slice(start, end))
            mu, qs = sub['mu'], sub['qs']
            n_valid = int(mu.count())

            u_val, a_val = get_u_a(ds_site, start_date=start, end_date=end)

            mu_neg = float(mu.where(mu < 0).sum())
            mu_pos = float(mu.where(mu > 0).sum())

            rows.append({
                'site': site_id,
                'time_window': f'{start} to {end}',
                'bearing': bearing_val,
                'n_valid': n_valid,
                'frac_masked': 1 - n_valid / int(mu.sizes['time']) if mu.sizes['time'] else np.nan,
                # min_count=1 so an all-NaN window stays NaN instead of collapsing to 0.0
                'mu_avg': float(mu.mean()) if n_valid else np.nan,
                'mu_frac_neg': float((mu < 0).sum() / n_valid) if n_valid else np.nan,
                'mu_neg_over_pos': abs(mu_neg / mu_pos) if mu_pos else np.nan,
                'qs_sum': float(qs.sum(min_count=1)),
                'u': float(u_val),
                'a': float(a_val),
            })
    return pd.DataFrame(rows)

In [13]:
def calc_stats_for_bearing(dp0, hs0, tp0, dp1, hs1, tp1, site_ids, bearing_val, time_windows, k, d=8):

    """Runs calc_qs_mu for both wave partitions at a single shoreline bearing and summarises
    each site/time window.

    qs, angle_diff and mu from both partitions are concatenated along time, so qs_sum, u, a and
    the mu diagnostics treat each partition's record as a separate wave contribution. mu_avg is
    the sum of each partition's own time-mean, so it matches the sum of the single-partition
    mu_avg values.
    """

    rows = []
    for site_id in site_ids:
        qs0_da, mu0_da, angle0_da = calc_qs_mu(
            dp=dp0.sel(site=site_id), hs=hs0.sel(site=site_id), tp=tp0.sel(site=site_id),
            angles=bearing_val, k=k, d=d,
        )
        qs1_da, mu1_da, angle1_da = calc_qs_mu(
            dp=dp1.sel(site=site_id), hs=hs1.sel(site=site_id), tp=tp1.sel(site=site_id),
            angles=bearing_val, k=k, d=d,
        )

        ds0 = xr.Dataset({'qs': qs0_da, 'angle_diff': angle0_da, 'mu': mu0_da})
        ds1 = xr.Dataset({'qs': qs1_da, 'angle_diff': angle1_da, 'mu': mu1_da})
        ds_site = xr.concat([ds0, ds1], dim='time').sortby('time')

        for start, end in time_windows:
            sub = ds_site.sel(time=slice(start, end))
            mu, qs = sub['mu'], sub['qs']
            n_valid = int(mu.count())

            mu0_win = ds0['mu'].sel(time=slice(start, end))
            mu1_win = ds1['mu'].sel(time=slice(start, end))
            mu_avg = float(mu0_win.mean()) + float(mu1_win.mean())

            u_val, a_val = get_u_a(ds_site, start_date=start, end_date=end)

            mu_neg = float(mu.where(mu < 0).sum())
            mu_pos = float(mu.where(mu > 0).sum())

            rows.append({
                'site': site_id,
                'time_window': f'{start} to {end}',
                'bearing': bearing_val,
                'n_valid': n_valid,
                'frac_masked': 1 - n_valid / int(mu.sizes['time']) if mu.sizes['time'] else np.nan,
                'mu_avg': mu_avg if n_valid else np.nan,
                'mu_frac_neg': float((mu < 0).sum() / n_valid) if n_valid else np.nan,
                'mu_neg_over_pos': abs(mu_neg / mu_pos) if mu_pos else np.nan,
                'qs_sum': float(qs.sum(min_count=1)),
                'u': float(u_val),
                'a': float(a_val),
            })
    return pd.DataFrame(rows)

In [10]:
dp0 = xr.open_dataset('dp0_NorthCarolina.nc') # peak direction 0
hs0 = xr.open_dataset('phs0_NorthCarolina.nc') # wave height 0
tp0 = xr.open_dataset('ptp0_NorthCarolina.nc') # peak period 0
dp1 = xr.open_dataset('dp1_NorthCarolina.nc') # peak direction 1
hs1 = xr.open_dataset('phs1_NorthCarolina.nc') # wave height 1
tp1 = xr.open_dataset('ptp1_NorthCarolina.nc') # peak period 1
shore = xr.open_dataset('NC_average_lat_lon.nc') # shoreline points

In [11]:
# compute shoreline angles (bearing from north)
angles_xr = get_angles_xr(shore)

In [14]:
# Statistics using the bearing between the ENDPOINTS of each domain
# HANNAH
id_list = [2844, 3096, 3142]

time_list = [
    ['1984-01-01T12:00:00', '2004-01-01T12:00:00'],
    ['2004-01-01T12:00:00', '2024-01-01T12:00:00'],
]

lat_bounds = [
    [35.237, 36.642],         
]

k_coeff = calc_alongshore_transport_k()

all_results = []
for lat_min, lat_max in lat_bounds:
    bearing_val, info = get_endpoint_bearing(angles_xr, lat_min, lat_max)

    print(f'[{lat_min}, {lat_max}]  n={info["n_sites"]}')
    print(f'  endpoints      : ({info["lat_start"]:.4f}, {info["lon_start"]:.4f}) '
          f'-> ({info["lat_end"]:.4f}, {info["lon_end"]:.4f})')
    print(f'  chord bearing  : {bearing_val:.2f} deg   seaward normal {info["seaward_normal"]:.2f} deg')
    print(f'  circular mean  : {info["circular_mean_bearing"]:.2f} deg '
          f'(chord - mean = {info["bearing_minus_mean"]:+.2f} deg)')
    print(f'  chord {info["chord_km"]:.1f} km / path {info["path_km"]:.1f} km  '
          f'-> sinuosity {info["sinuosity"]:.3f}\n')

    df = calc_stats_for_bearing(dp0, hs0, tp0, dp1, hs1, tp1, id_list, bearing_val, time_list, k_coeff)
    df.insert(1, 'lat_bounds', str([lat_min, lat_max]))
    all_results.append(df)

df_endpoint = pd.concat(all_results, ignore_index=True)
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(df_endpoint.to_string(index=False))

[35.237, 36.642]  n=2971
  endpoints      : (35.2371, -75.5262) -> (36.5504, -75.8674)
  chord bearing  : 348.21 deg   seaward normal 78.21 deg
  circular mean  : 348.32 deg (chord - mean = -0.11 deg)
  chord 149.2 km / path 155.8 km  -> sinuosity 1.044

 site       lat_bounds                                time_window    bearing  n_valid  frac_masked    mu_avg  mu_frac_neg  mu_neg_over_pos      qs_sum        u        a
 2844 [35.237, 36.642] 1984-01-01T12:00:00 to 2004-01-01T12:00:00 348.213619   216491     0.382622  0.004439     0.488949         0.662146 6445.666504 0.419403 0.762037
 2844 [35.237, 36.642] 2004-01-01T12:00:00 to 2024-01-01T12:00:00 348.213619   214829     0.387281  0.007067     0.484976         0.597114 6598.966797 0.400556 0.753793
 3096 [35.237, 36.642] 1984-01-01T12:00:00 to 2004-01-01T12:00:00 348.213619   222433     0.365677 -0.001971     0.499251         0.931068 6159.658691 0.451596 0.731238
 3096 [35.237, 36.642] 2004-01-01T12:00:00 to 2024-01-01T12:00:00 348

In [15]:
# Statistics using the bearing between the ENDPOINTS of each domain
# ROYA
id_list = [2844, 3096, 3142]

time_list = [
    ['1992-01-01T12:00:00', '2007-01-01T12:00:00'],
    ['2008-01-01T12:00:00', '2024-01-01T12:00:00'],
]

lat_bounds = [
    [35.231204, 35.774414]       
]

k_coeff = calc_alongshore_transport_k()

all_results = []
for lat_min, lat_max in lat_bounds:
    bearing_val, info = get_endpoint_bearing(angles_xr, lat_min, lat_max)

    print(f'[{lat_min}, {lat_max}]  n={info["n_sites"]}')
    print(f'  endpoints      : ({info["lat_start"]:.4f}, {info["lon_start"]:.4f}) '
          f'-> ({info["lat_end"]:.4f}, {info["lon_end"]:.4f})')
    print(f'  chord bearing  : {bearing_val:.2f} deg   seaward normal {info["seaward_normal"]:.2f} deg')
    print(f'  circular mean  : {info["circular_mean_bearing"]:.2f} deg '
          f'(chord - mean = {info["bearing_minus_mean"]:+.2f} deg)')
    print(f'  chord {info["chord_km"]:.1f} km / path {info["path_km"]:.1f} km  '
          f'-> sinuosity {info["sinuosity"]:.3f}\n')

    df = calc_stats_for_bearing(dp0, hs0, tp0, dp1, hs1, tp1, id_list, bearing_val, time_list, k_coeff)
    df.insert(1, 'lat_bounds', str([lat_min, lat_max]))
    all_results.append(df)

df_endpoint = pd.concat(all_results, ignore_index=True)
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(df_endpoint.to_string(index=False))

[35.231204, 35.774414]  n=1330
  endpoints      : (35.2312, -75.6104) -> (35.7731, -75.5245)
  chord bearing  : 7.33 deg   seaward normal 97.33 deg
  circular mean  : 4.77 deg (chord - mean = +2.55 deg)
  chord 60.8 km / path 289.8 km  -> sinuosity 4.769

 site             lat_bounds                                time_window  bearing  n_valid  frac_masked    mu_avg  mu_frac_neg  mu_neg_over_pos      qs_sum        u        a
 2844 [35.231204, 35.774414] 1992-01-01T12:00:00 to 2007-01-01T12:00:00 7.325484   178455     0.321448 -0.014484     0.480861         1.898562 4494.393066 0.510617 0.715453
 2844 [35.231204, 35.774414] 2008-01-01T12:00:00 to 2024-01-01T12:00:00 7.325484   191305     0.317957 -0.014566     0.484697         1.806975 4923.423340 0.516379 0.713441
 3096 [35.231204, 35.774414] 1992-01-01T12:00:00 to 2007-01-01T12:00:00 7.325484   184626     0.297984 -0.014348     0.504306         1.779161 3880.896973 0.489111 0.670815
 3096 [35.231204, 35.774414] 2008-01-01T12:00:00 to 

In [16]:
# HANNAH SINGLE 0
id_list = [2844, 3096, 3142]

time_list = [
    ['1984-01-01T12:00:00', '2004-01-01T12:00:00'],
    ['2004-01-01T12:00:00', '2024-01-01T12:00:00'],
]

lat_bounds = [
    [35.237, 36.642],         
]

k_coeff = calc_alongshore_transport_k()

all_results = []
for lat_min, lat_max in lat_bounds:
    bearing_val, info = get_endpoint_bearing(angles_xr, lat_min, lat_max)

    print(f'[{lat_min}, {lat_max}]  n={info["n_sites"]}')
    print(f'  endpoints      : ({info["lat_start"]:.4f}, {info["lon_start"]:.4f}) '
          f'-> ({info["lat_end"]:.4f}, {info["lon_end"]:.4f})')
    print(f'  chord bearing  : {bearing_val:.2f} deg   seaward normal {info["seaward_normal"]:.2f} deg')
    print(f'  circular mean  : {info["circular_mean_bearing"]:.2f} deg '
          f'(chord - mean = {info["bearing_minus_mean"]:+.2f} deg)')
    print(f'  chord {info["chord_km"]:.1f} km / path {info["path_km"]:.1f} km  '
          f'-> sinuosity {info["sinuosity"]:.3f}\n')

    df = calc_stats_for_bearing_single(dp0, hs0, tp0, id_list, bearing_val, time_list, k_coeff)
    df.insert(1, 'lat_bounds', str([lat_min, lat_max]))
    all_results.append(df)

df_endpoint = pd.concat(all_results, ignore_index=True)
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(df_endpoint.to_string(index=False))

[35.237, 36.642]  n=2971
  endpoints      : (35.2371, -75.5262) -> (36.5504, -75.8674)
  chord bearing  : 348.21 deg   seaward normal 78.21 deg
  circular mean  : 348.32 deg (chord - mean = -0.11 deg)
  chord 149.2 km / path 155.8 km  -> sinuosity 1.044

 site       lat_bounds                                time_window    bearing  n_valid  frac_masked    mu_avg  mu_frac_neg  mu_neg_over_pos      qs_sum        u        a
 2844 [35.237, 36.642] 1984-01-01T12:00:00 to 2004-01-01T12:00:00 348.213619    72357     0.587312  0.001193     0.601628         0.911823 6980.009766 0.457179 0.873095
 2844 [35.237, 36.642] 2004-01-01T12:00:00 to 2024-01-01T12:00:00 348.213619    71358     0.592956  0.003183     0.595659         0.807191 7139.164062 0.442658 0.872438
 3096 [35.237, 36.642] 1984-01-01T12:00:00 to 2004-01-01T12:00:00 348.213619    73584     0.580314 -0.005506     0.642246         1.412238 6443.746582 0.505950 0.830753
 3096 [35.237, 36.642] 2004-01-01T12:00:00 to 2024-01-01T12:00:00 348

In [17]:
# HANNAH SINGLE 1
id_list = [2844, 3096, 3142]

time_list = [
    ['1984-01-01T12:00:00', '2004-01-01T12:00:00'],
    ['2004-01-01T12:00:00', '2024-01-01T12:00:00'],
]

lat_bounds = [
    [35.237, 36.642],         
]

k_coeff = calc_alongshore_transport_k()

all_results = []
for lat_min, lat_max in lat_bounds:
    bearing_val, info = get_endpoint_bearing(angles_xr, lat_min, lat_max)

    print(f'[{lat_min}, {lat_max}]  n={info["n_sites"]}')
    print(f'  endpoints      : ({info["lat_start"]:.4f}, {info["lon_start"]:.4f}) '
          f'-> ({info["lat_end"]:.4f}, {info["lon_end"]:.4f})')
    print(f'  chord bearing  : {bearing_val:.2f} deg   seaward normal {info["seaward_normal"]:.2f} deg')
    print(f'  circular mean  : {info["circular_mean_bearing"]:.2f} deg '
          f'(chord - mean = {info["bearing_minus_mean"]:+.2f} deg)')
    print(f'  chord {info["chord_km"]:.1f} km / path {info["path_km"]:.1f} km  '
          f'-> sinuosity {info["sinuosity"]:.3f}\n')

    df = calc_stats_for_bearing_single(dp1, hs1, tp1, id_list, bearing_val, time_list, k_coeff)
    df.insert(1, 'lat_bounds', str([lat_min, lat_max]))
    all_results.append(df)

df_endpoint = pd.concat(all_results, ignore_index=True)
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(df_endpoint.to_string(index=False))

[35.237, 36.642]  n=2971
  endpoints      : (35.2371, -75.5262) -> (36.5504, -75.8674)
  chord bearing  : 348.21 deg   seaward normal 78.21 deg
  circular mean  : 348.32 deg (chord - mean = -0.11 deg)
  chord 149.2 km / path 155.8 km  -> sinuosity 1.044

 site       lat_bounds                                time_window    bearing  n_valid  frac_masked   mu_avg  mu_frac_neg  mu_neg_over_pos      qs_sum        u        a
 2844 [35.237, 36.642] 1984-01-01T12:00:00 to 2004-01-01T12:00:00 348.213619   144134     0.177932 0.003246     0.432382         0.292774 -534.342041 0.299413 0.409278
 2844 [35.237, 36.642] 2004-01-01T12:00:00 to 2024-01-01T12:00:00 348.213619   143471     0.181606 0.003884     0.429927         0.275170 -540.196960 0.282441 0.420939
 3096 [35.237, 36.642] 1984-01-01T12:00:00 to 2004-01-01T12:00:00 348.213619   148849     0.151040 0.003535     0.428562         0.319167 -284.090088 0.303608 0.460298
 3096 [35.237, 36.642] 2004-01-01T12:00:00 to 2024-01-01T12:00:00 348.213

In [18]:
# ROYA SINGLE 0
id_list = [2844, 3096, 3142]

time_list = [
    ['1992-01-01T12:00:00', '2007-01-01T12:00:00'],
    ['2008-01-01T12:00:00', '2024-01-01T12:00:00'],
]

lat_bounds = [
    [35.231204, 35.774414]       
]

k_coeff = calc_alongshore_transport_k()

all_results = []
for lat_min, lat_max in lat_bounds:
    bearing_val, info = get_endpoint_bearing(angles_xr, lat_min, lat_max)

    print(f'[{lat_min}, {lat_max}]  n={info["n_sites"]}')
    print(f'  endpoints      : ({info["lat_start"]:.4f}, {info["lon_start"]:.4f}) '
          f'-> ({info["lat_end"]:.4f}, {info["lon_end"]:.4f})')
    print(f'  chord bearing  : {bearing_val:.2f} deg   seaward normal {info["seaward_normal"]:.2f} deg')
    print(f'  circular mean  : {info["circular_mean_bearing"]:.2f} deg '
          f'(chord - mean = {info["bearing_minus_mean"]:+.2f} deg)')
    print(f'  chord {info["chord_km"]:.1f} km / path {info["path_km"]:.1f} km  '
          f'-> sinuosity {info["sinuosity"]:.3f}\n')

    df = calc_stats_for_bearing_single(dp0, hs0, tp0, id_list, bearing_val, time_list, k_coeff)
    df.insert(1, 'lat_bounds', str([lat_min, lat_max]))
    all_results.append(df)

df_endpoint = pd.concat(all_results, ignore_index=True)
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(df_endpoint.to_string(index=False))

[35.231204, 35.774414]  n=1330
  endpoints      : (35.2312, -75.6104) -> (35.7731, -75.5245)
  chord bearing  : 7.33 deg   seaward normal 97.33 deg
  circular mean  : 4.77 deg (chord - mean = +2.55 deg)
  chord 60.8 km / path 289.8 km  -> sinuosity 4.769

 site             lat_bounds                                time_window  bearing  n_valid  frac_masked    mu_avg  mu_frac_neg  mu_neg_over_pos      qs_sum        u        a
 2844 [35.231204, 35.774414] 1992-01-01T12:00:00 to 2007-01-01T12:00:00 7.325484    59986     0.543822 -0.016905     0.764112         3.997475 3948.086426 0.598464 0.762505
 2844 [35.231204, 35.774414] 2008-01-01T12:00:00 to 2024-01-01T12:00:00 7.325484    63879     0.544515 -0.017037     0.768766         3.548272 4376.737793 0.610004 0.767897
 3096 [35.231204, 35.774414] 1992-01-01T12:00:00 to 2007-01-01T12:00:00 7.325484    58881     0.552226 -0.016630     0.757375         3.509018 3030.914062 0.591482 0.698417
 3096 [35.231204, 35.774414] 2008-01-01T12:00:00 to 

In [19]:
# ROYA SINGLE 1
id_list = [2844, 3096, 3142]

time_list = [
    ['1992-01-01T12:00:00', '2007-01-01T12:00:00'],
    ['2008-01-01T12:00:00', '2024-01-01T12:00:00'],
]

lat_bounds = [
    [35.231204, 35.774414]       
]

k_coeff = calc_alongshore_transport_k()

all_results = []
for lat_min, lat_max in lat_bounds:
    bearing_val, info = get_endpoint_bearing(angles_xr, lat_min, lat_max)

    print(f'[{lat_min}, {lat_max}]  n={info["n_sites"]}')
    print(f'  endpoints      : ({info["lat_start"]:.4f}, {info["lon_start"]:.4f}) '
          f'-> ({info["lat_end"]:.4f}, {info["lon_end"]:.4f})')
    print(f'  chord bearing  : {bearing_val:.2f} deg   seaward normal {info["seaward_normal"]:.2f} deg')
    print(f'  circular mean  : {info["circular_mean_bearing"]:.2f} deg '
          f'(chord - mean = {info["bearing_minus_mean"]:+.2f} deg)')
    print(f'  chord {info["chord_km"]:.1f} km / path {info["path_km"]:.1f} km  '
          f'-> sinuosity {info["sinuosity"]:.3f}\n')

    df = calc_stats_for_bearing_single(dp1, hs1, tp1, id_list, bearing_val, time_list, k_coeff)
    df.insert(1, 'lat_bounds', str([lat_min, lat_max]))
    all_results.append(df)

df_endpoint = pd.concat(all_results, ignore_index=True)
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(df_endpoint.to_string(index=False))

[35.231204, 35.774414]  n=1330
  endpoints      : (35.2312, -75.6104) -> (35.7731, -75.5245)
  chord bearing  : 7.33 deg   seaward normal 97.33 deg
  circular mean  : 4.77 deg (chord - mean = +2.55 deg)
  chord 60.8 km / path 289.8 km  -> sinuosity 4.769

 site             lat_bounds                                time_window  bearing  n_valid  frac_masked   mu_avg  mu_frac_neg  mu_neg_over_pos     qs_sum        u        a
 2844 [35.231204, 35.774414] 1992-01-01T12:00:00 to 2007-01-01T12:00:00 7.325484   118469     0.099075 0.002421     0.337438         0.391185 546.306824 0.283610 0.593864
 2844 [35.231204, 35.774414] 2008-01-01T12:00:00 to 2024-01-01T12:00:00 7.325484   127426     0.091398 0.002471     0.342293         0.407598 546.685242 0.289085 0.581236
 3096 [35.231204, 35.774414] 1992-01-01T12:00:00 to 2007-01-01T12:00:00 7.325484   125745     0.043742 0.002282     0.385805         0.423989 849.982971 0.279056 0.614176
 3096 [35.231204, 35.774414] 2008-01-01T12:00:00 to 2024-01-